In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import itertools
from ast import literal_eval

In [2]:
df = pd.read_csv("data/tbi_full.csv")
df

,wave,person_id,trip_id,travel_date,travel_dow,depart_time,arrive_time,duration,distance,vehicle_trips,...,senior,student,unemployed,parent,person_type,gender_cleaned,o_car_terminal_time,d_car_terminal_time,car_duration_seconds_adj,bike_duration_seconds_adj
0,1,1811206201,[1811206201001],2018-12-06,Thursday,12:30:00,13:00:00,23.0,16.443340,1.0,...,False,False,False,True,working adult with kids,Female,1.0,1.0,1335.6,14862.5
1,1,1811206201,[1811206201002],2018-12-06,Thursday,22:00:00,22:30:00,22.0,16.462600,1.0,...,False,False,False,True,working adult with kids,Female,1.0,1.0,1337.4,14862.5
2,1,1811206203,[1811206203001],2018-12-06,Thursday,05:45:00,06:45:00,14.0,9.543020,0.0,...,False,True,True,True,non-working adult with kids,Male,1.0,1.0,1065.6,6754.2
3,1,1811206203,[1811206203002],2018-12-06,Thursday,14:00:00,15:00:00,14.0,9.546120,0.0,...,False,True,True,True,non-working adult with kids,Male,1.0,1.0,1126.5,6754.2
4,1,1811206204,[1811206204001],2018-12-06,Thursday,10:45:00,11:15:00,12.0,8.328860,0.0,...,False,True,True,True,non-working adult with kids,Male,1.0,1.0,943.5,6490.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
454176,2,2200279003,[2200279003003],2022-01-27,Thursday,23:13:00,23:34:00,21.0,1.694825,1.0,...,False,False,False,True,working adult with kids,Male,1.0,3.0,670.7,1678.2
454177,2,2200279101,[2200279101001],2022-01-26,Wednesday,14:06:00,14:26:00,20.0,4.468702,1.0,...,False,False,False,False,working adult without kids,Female,1.0,1.0,754.0,3478.2
454178,2,2200279101,[2200279101002],2022-01-26,Wednesday,23:03:00,23:29:00,26.0,2.566255,1.0,...,False,False,False,False,working adult without kids,Female,1.0,1.0,741.4,3478.2
454179,2,2200279102,[2200279102001],2022-01-26,Wednesday,13:50:00,14:15:00,25.0,1.760551,0.0,...,False,False,False,False,working adult without kids,Male,1.0,1.0,532.4,2372.2


In [3]:
keep_cols = ["wave", "person_id", "trip_id", "travel_date", "depart_time", "arrive_time", "duration", "distance", "vehicle_trips", "vmt", "mode", "o_purpose_category", "d_purpose_category", "income_detailed", "temperature", "precipitation", "snow_depth", "car_distance_meters", "walk_duration_seconds", "walk_distance_meters", "bike_distance_meters", "bike_distance_meters_1", "bike_distance_meters_2", "bike_distance_meters_3", "bike_distance_meters_4", "transit_duration", "transit_access_length", "transit_num_transfers", "community", "purpose_cleaned", "person_type", "gender_cleaned", "car_duration_seconds_adj", "bike_duration_seconds_adj"]
df_anonymous = df[keep_cols].copy()

In [4]:
df_anonymous["person_id"] = pd.factorize(df["person_id"])[0]
df_anonymous["person_id"] = df_anonymous["person_id"].astype("Int32")

In [5]:
trips = list(itertools.chain(*df["trip_id"].apply(lambda x: literal_eval(str(x))).values))
temp = pd.factorize(trips)
trip_conversion = dict(zip(temp[1], temp[0]))

In [6]:
def map_trip_ids(x: list) -> list:
    res = list(map(lambda ele: trip_conversion[ele], x))
    return res

df_anonymous["trip_id"] = df["trip_id"].apply(lambda x: literal_eval(x)).apply(lambda x: map_trip_ids(x))

In [7]:
df_anonymous["wave"] = df["wave"].astype("Int8")
# df_anonymous["person_id"] = df_anonymous["person_id"].astype("Int32")
df_anonymous["duration"] = df["duration"].round().astype("Int16")
df_anonymous["distance"] = df["distance"].round(1).astype("Float32")
df_anonymous["vehicle_trips"] = df["vehicle_trips"].astype("Float32")
df_anonymous["vmt"] = df["vmt"].astype("Float32")
df_anonymous["temperature"] = df["temperature"].astype("Float32")
df_anonymous["precipitation"] = df["precipitation"].astype("Float32")
df_anonymous["snow_depth"] = df["snow_depth"].astype("Float32")
df_anonymous["car_distance_meters"] = df["car_distance_meters"].round().astype("Int32")
df_anonymous["walk_duration_seconds"] = df["walk_duration_seconds"].round().astype("Int32")
df_anonymous["walk_distance_meters"] = df["walk_distance_meters"].round().astype("Int32")
df_anonymous["bike_distance_meters"] = df["bike_distance_meters"].round().astype("Int32")
df_anonymous["bike_distance_meters_1"] = df["bike_distance_meters_1"].round().astype("Int32")
df_anonymous["bike_distance_meters_2"] = df["bike_distance_meters_2"].round().astype("Int32")
df_anonymous["bike_distance_meters_3"] = df["bike_distance_meters_3"].round().astype("Int32")
df_anonymous["bike_distance_meters_4"] = df["bike_distance_meters_4"].round().astype("Int32")
df_anonymous["transit_duration"] = df["transit_duration"].round().astype("Int16")
df_anonymous["transit_access_length"] = df["transit_access_length"].round().astype("Int16")
df_anonymous["transit_num_transfers"] = df["transit_num_transfers"].round().astype("Int8")
df_anonymous["car_duration_seconds_adj"] = df["car_duration_seconds_adj"].round(1).astype("Float32")
df_anonymous["bike_duration_seconds_adj"] = df["bike_duration_seconds_adj"].round(1).astype("Float32")

In [8]:
df_anonymous.to_parquet("data/tbi_anonymous.parquet")